In [ ]:
!git clone https://github.com/rm-a0/3d-nca --quiet
%cd /kaggle/working/3d-nca
!pip install pyvista

In [ ]:
import os, sys, random
from collections import Counter
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
from typing import Any, Dict, Generator
 
sys.path.insert(0, '/kaggle/working/3d-nca')
 
from src.core.cell          import CellConfig
from src.core.grid          import Grid3D, GridConfig
from src.core.perception    import PerceptionConfig
from src.core.update        import UpdateConfig
from src.core.nca_model     import NCAModel, NCAConfig
from src.core.schedule      import Event, EventType, Schedule
from src.core.runners.base  import NCARunner, TrainingSnapshot
from src.server.logger      import NCALogger
from src.viz.volume_mpl     import show_volume_rgba_mpl, show_state_rgba_mpl

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")
if device == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
TARGET_PATHS = [
    "/kaggle/input/datasets/michalrepk/3d-nca-pokemon/charmander.npz",
    "/kaggle/input/datasets/michalrepk/3d-nca-pokemon/charmeleon.npz",
    "/kaggle/input/datasets/michalrepk/3d-nca-pokemon/charizard.npz",
]
TASK_NAMES  = ["Creature 1", "Creature 2", "Creature 3"]
PHASE_NAMES = [
    f"Seed -> {TASK_NAMES[0]}",
    f"{TASK_NAMES[0]} -> {TASK_NAMES[1]}",
    f"{TASK_NAMES[1]} -> {TASK_NAMES[2]}",
]
OUTPUT_DIR = "/kaggle/working/nca_morph"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
def load_npz_target(path, rotate_k=1):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Target not found: {path}")
    with np.load(path, allow_pickle=True) as data:
        arr = data["target"].astype(np.float32)
    if arr.ndim != 4 or arr.shape[-1] != 4:
        raise ValueError(f"Expected (D,H,W,4), got {arr.shape} in {path}")
    if rotate_k != 0:
        arr = np.rot90(arr, k=rotate_k, axes=(0, 1))
    return arr
 
print("Loading targets...")
TARGETS   = [load_npz_target(p) for p in TARGET_PATHS]
GRID_SIZE = TARGETS[0].shape[:3]
N_TASKS   = len(TARGETS)
 
if len({t.shape[:3] for t in TARGETS}) != 1:
    raise ValueError("All targets must share the same grid size.")
 
print(f"Grid size : {GRID_SIZE}")
for name, t in zip(TASK_NAMES, TARGETS):
    print(f"  {name:12s}: {int((t[..., 3] > 0.1).sum()):>5} alive voxels")

In [ ]:
plt.rcParams.update({"figure.dpi": 130, "font.size": 9, "figure.facecolor": "white"})
 
fig, axes = plt.subplots(1, N_TASKS, figsize=(N_TASKS * 4.5, 5.0),
                         subplot_kw={"projection": "3d"})
for col, (rgba, name) in enumerate(zip(TARGETS, TASK_NAMES)):
    n = show_volume_rgba_mpl(rgba, ax=axes[col], surface_only=True,
                             threshold=0.15, point_size=30,
                             view_angle=(28, 40), show=False, title="")
    axes[col].set_title(f"{name}\n({n} surface voxels)", fontsize=9)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/targets.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
PHASE_0_EPOCHS = 1500
PHASE_1_EPOCHS = 6000
PHASE_2_EPOCHS = 12000
TOTAL_EPOCHS   = PHASE_0_EPOCHS + PHASE_1_EPOCHS + PHASE_2_EPOCHS
 
BOUNDARY_1 = PHASE_0_EPOCHS + 1
BOUNDARY_2 = PHASE_0_EPOCHS + PHASE_1_EPOCHS + 1
 
config = {
    "cell": {
        "hidden_channels":  16,
        "visible_channels": 4,
        "alive_threshold":  0.1,
        "task_channels":    N_TASKS,
        "pos_channels":     3,
    },
    "perception": {
        "kernel_radius":  1,
        "channel_groups": 5,
    },
    "update": {
        "hidden_dim":        128,
        "stochastic_update": False,
        "fire_rate":         0.5,
    },
    "grid": {
        "size": GRID_SIZE,
    },
    "training": {
        "learning_rate":   2e-3,
        "batch_size":      4,
        "pool_size":       64,
        "total_epochs":    TOTAL_EPOCHS,
        "phase_0_epochs":  PHASE_0_EPOCHS,
        # Initial weights -- all higher than v1 based on loss curve analysis:
        # color_weight: color plateau at 1e-2 in phases 1+2 was the main failure
        # overflow_weight: too weak to clear predecessor voxels (black blob)
        "alpha_weight":    3.0,
        "color_weight":    2.0,
        "overflow_weight": 3.0,
    },
}

In [ ]:
POOL_EXPAND = "POOL_EXPAND"  # new event type; compatible with str-based EventType
 
 
class MetamorphosisRunner(NCARunner):
    """Multi-task metamorphogenesis runner with curriculum joint training.
 
    Implements NCARunner so it works with Schedule, NCATrainer, and NCAServer
    without modification. Phase transitions are driven entirely by schedule
    events through on_event() -- no epoch checks in the training loop.
 
    Pool management
    ---------------
    The pool is a replay buffer keeping states from all active tasks alive
    so every batch contains gradient signal for every task simultaneously.
    Expansion at phase boundaries converts the best-converged predecessor
    slots into seeds for the next task. Best means lowest MSE against the
    predecessor target -- these are the most realistic morphing inputs.
 
    Schedule events handled
    -----------------------
    LEARNING_RATE, ALPHA_WEIGHT, COLOR_WEIGHT, OVERFLOW_WEIGHT (built-in),
    POOL_EXPAND (new -- drives pool expansion without epoch hardcoding).
    """
 
    def __init__(self):
        self.model           = None
        self.optimizer       = None
        self.targets_t       = []
        self.pool            = []
        self.pool_tasks      = []
        self.W               = {}
        self._cell_cfg       = None
        self._device         = None
        self._batch_size     = 4
        self._phase_0_epochs = 2000
        self.state           = None
        self.current_epoch   = 0
        self.total_epochs    = 0
        self.latest_loss     = 0.0
        self._latest_metrics: Dict[str, float] = {}
 
    # -- NCARunner interface --------------------------------------------------
 
    def init(self, config: dict, target) -> None:
        if not isinstance(target, list):
            target = [target]
 
        self._cell_cfg = CellConfig(**config["cell"])
        perc_cfg       = PerceptionConfig(**config["perception"])
        upd_cfg        = UpdateConfig(**config["update"])
        grid_cfg       = GridConfig(**config["grid"])
        self._device   = "cuda" if torch.cuda.is_available() else "cpu"
 
        self.model = Grid3D(
            self._cell_cfg, perc_cfg, upd_cfg, grid_cfg
        ).to(self._device)
 
        self.optimizer = torch.optim.AdamW(
            self.model.parameters(),
            lr=config["training"]["learning_rate"],
            weight_decay=1e-5,
        )
 
        self.targets_t       = [self._prep_target(t) for t in target]
        pool_size            = config["training"].get("pool_size", 32)
        self.pool            = [self._new_seed(0) for _ in range(pool_size)]
        self.pool_tasks      = [0] * pool_size
        self.total_epochs    = config["training"]["total_epochs"]
        self._batch_size     = config["training"].get("batch_size", 4)
        self._phase_0_epochs = config["training"].get("phase_0_epochs", 2000)
        self.W = {
            "alpha":    config["training"].get("alpha_weight",    3.0),
            "color":    config["training"].get("color_weight",    2.0),
            "overflow": config["training"].get("overflow_weight", 3.0),
        }
        self.current_epoch   = 0
        self.latest_loss     = 0.0
        self._latest_metrics = {}
 
    def train(self, schedule=None) -> Generator[Dict[str, Any], None, None]:
        for epoch in range(1, self.total_epochs + 1):
            if schedule is not None:
                schedule.check_and_execute(epoch, self)
            metrics = self._step(epoch)
            self.current_epoch   = epoch
            self.latest_loss     = metrics["loss_total"]
            self._latest_metrics = {
                k: float(v) for k, v in metrics.items()
                if isinstance(v, (int, float))
            }
            yield metrics
 
    def snapshot(self) -> TrainingSnapshot:
        return TrainingSnapshot(
            state            = self.state.detach().cpu().numpy().astype(np.float32),
            epoch            = self.current_epoch,
            total_epochs     = self.total_epochs,
            loss             = self.latest_loss,
            visible_channels = self._cell_cfg.visible_channels,
            metrics          = dict(self._latest_metrics),
        )
 
    def set_target(self, target) -> None:
        """Not applicable: targets are fixed at init for multi-task training."""
        pass
 
    def on_event(self, event) -> bool:
        t = str(event.event_type)
        v = float(event.value)
 
        if t == "LEARNING_RATE":
            for pg in self.optimizer.param_groups:
                pg["lr"] = v
            print(f"  [Runner] LR -> {v:.2e}")
            return True
 
        elif t == "ALPHA_WEIGHT":
            self.W["alpha"] = v
            print(f"  [Runner] alpha_weight -> {v}")
            return True
 
        elif t == "COLOR_WEIGHT":
            self.W["color"] = v
            print(f"  [Runner] color_weight -> {v}")
            return True
 
        elif t == "OVERFLOW_WEIGHT":
            self.W["overflow"] = v
            print(f"  [Runner] overflow_weight -> {v}")
            return True
 
        elif t == POOL_EXPAND:
            self._expand_pool(int(v))
            return True
 
        return False
 
    # -- Private helpers ------------------------------------------------------
 
    def _expand_pool(self, to_task: int) -> None:
        from_task = to_task - 1
        vis       = self._cell_cfg.visible_channels
        from_idx  = [i for i, x in enumerate(self.pool_tasks) if x == from_task]
        n_convert = len(from_idx) // 2
        tgt       = self.targets_t[from_task]
 
        with torch.no_grad():
            scored = sorted(
                [(F.mse_loss(self.pool[i][:, -vis:], tgt[:, -vis:]).item(), i)
                 for i in from_idx]
            )
        for _, idx in scored[:n_convert]:
            self.pool[idx]       = self._relabel(self.pool[idx], to_task)
            self.pool_tasks[idx] = to_task
 
        dist = dict(Counter(self.pool_tasks))
        print(f"  [Runner] POOL_EXPAND task {from_task} -> {to_task}  pool: {dist}")
 
    def _step(self, epoch: int) -> Dict[str, Any]:
        vis    = self._cell_cfg.visible_channels
        bs     = min(len(self.pool), self._batch_size)
        lo, hi = self._step_range(epoch)
        n_steps = random.randint(lo, hi)
 
        indices    = random.sample(range(len(self.pool)), bs)
        batch      = torch.cat([self.pool[i] for i in indices], dim=0)
        task_ids_b = [self.pool_tasks[i] for i in indices]
        tgt_batch  = torch.cat([self.targets_t[t] for t in task_ids_b], dim=0)
 
        with torch.no_grad():
            per_loss = [
                F.mse_loss(batch[i:i+1, -vis:], self.targets_t[task_ids_b[i]][:, -vis:]).item()
                for i in range(bs)
            ]
            worst_b    = int(np.argmax(per_loss))
            worst_task = task_ids_b[worst_b]
            if worst_task == 0:
                batch[worst_b:worst_b+1] = self._new_seed(0)
            else:
                parent_idx = [i for i, t in enumerate(self.pool_tasks)
                              if t == worst_task - 1]
                if parent_idx:
                    src = random.choice(parent_idx)
                    batch[worst_b:worst_b+1] = self._relabel(self.pool[src], worst_task)
                else:
                    batch[worst_b:worst_b+1] = self._new_seed(worst_task)
 
        self.optimizer.zero_grad()
        state    = self.model(batch, steps=n_steps)
        pred_vis = state[:, -vis:]
        tgt_vis  = tgt_batch[:, -vis:]
        tgt_a    = tgt_vis[:, -1:]
        tgt_mask = (tgt_a > self._cell_cfg.alive_threshold).float()
 
        loss_alpha    = F.mse_loss(pred_vis[:, -1:], tgt_a)
        color_diff    = (pred_vis[:, :-1] - tgt_vis[:, :-1]) ** 2 * tgt_mask
        loss_color    = color_diff.sum() / tgt_mask.sum().clamp(min=1.0)
        loss_overflow = ((pred_vis[:, -1:] * (1.0 - tgt_mask)) ** 2).mean()
        loss = (self.W["alpha"]    * loss_alpha
              + self.W["color"]    * loss_color
              + self.W["overflow"] * loss_overflow)
 
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
        self.optimizer.step()
 
        with torch.no_grad():
            for j, idx in enumerate(indices):
                self.pool[idx] = state[j:j+1].detach()
        self.state = state[0:1].detach()
 
        return {
            "loss_total":    loss.item(),
            "loss_alpha":    loss_alpha.item(),
            "loss_color":    loss_color.item(),
            "loss_overflow": loss_overflow.item(),
        }
 
    def _step_range(self, epoch: int):
        """Curriculum: NCA steps ramp (8,16) -> (32,64) over phase_0_epochs."""
        t = min(epoch / self._phase_0_epochs, 1.0)
        return max(int(8 + t * 24), 4), max(int(16 + t * 48), 5)
 
    def _new_seed(self, task_id: int):
        return self.model.seed_center(
            1, self._device, torch.tensor([task_id], device=self._device)
        )
 
    def _relabel(self, state, task_id: int):
        tc  = self._cell_cfg.task_channels
        vis = self._cell_cfg.visible_channels
        s   = state.clone()
        one_hot = F.one_hot(
            torch.tensor([task_id], device=s.device), num_classes=tc
        ).float()
        s[:, -(vis + tc) : -vis, ...] = (
            one_hot.view(1, tc, 1, 1, 1).expand(-1, -1, *s.shape[2:])
        )
        return s
 
    def _prep_target(self, rgba: np.ndarray):
        t = np.transpose(rgba, (3, 0, 1, 2)).astype(np.float32)
        return torch.from_numpy(t).unsqueeze(0).to(self._device)

In [ ]:
schedule = Schedule()

# --- Phase 1: Complexity Jump ---
schedule.add_event(Event(epoch=BOUNDARY_1,        event_type=POOL_EXPAND,               value=1.0))
schedule.add_event(Event(epoch=BOUNDARY_1,        event_type=EventType.LEARNING_RATE,   value=8e-4))
schedule.add_event(Event(epoch=BOUNDARY_1,        event_type=EventType.COLOR_WEIGHT,    value=3.0))
schedule.add_event(Event(epoch=BOUNDARY_1 + 2000, event_type=EventType.OVERFLOW_WEIGHT, value=4.0)) 
 
# --- Phase 2: The Final Polish ---
schedule.add_event(Event(epoch=BOUNDARY_2,        event_type=POOL_EXPAND,               value=2.0))
schedule.add_event(Event(epoch=BOUNDARY_2,        event_type=EventType.LEARNING_RATE,   value=3e-4)) 
schedule.add_event(Event(epoch=BOUNDARY_2,        event_type=EventType.COLOR_WEIGHT,    value=5.0)) 

# At epoch 15,000, we drop LR even lower and hike overflow weight 
# to force the model to delete stray floating voxels.
schedule.add_event(Event(epoch=15000,             event_type=EventType.LEARNING_RATE,   value=1e-4))
schedule.add_event(Event(epoch=15000,             event_type=EventType.OVERFLOW_WEIGHT, value=8.0))

print(f"Schedule ({len(schedule.events)} events):")
for ev in sorted(schedule.events, key=lambda e: e.epoch):
    print(f"  epoch {ev.epoch:>6}: {str(ev.event_type):<22} value={ev.value}")
 
# -- 8. Logger ----------------------------------------------------------------
 
logger = NCALogger(base_dir=OUTPUT_DIR, checkpoint_interval=500)
logger.log_meta(config)
print(f"\nRun dir: {logger.run_dir}")

In [ ]:
runner = MetamorphosisRunner()
runner.init(config, TARGETS)
 
params = sum(p.numel() for p in runner.model.parameters() if p.requires_grad)
print(f"\nRunner ready -- {params:,} trainable params")
print(f"  hidden_dim={config['update']['hidden_dim']}  "
      f"pool={config['training']['pool_size']}  "
      f"total_epochs={TOTAL_EPOCHS}")
print(f"  Phase 0: epochs 1-{PHASE_0_EPOCHS}  (task 0 solo)")
print(f"  Phase 1: epochs {BOUNDARY_1}-{BOUNDARY_2-1}  (tasks 0+1 joint)")
print(f"  Phase 2: epochs {BOUNDARY_2}-{TOTAL_EPOCHS}  (all 3 tasks joint)")
print()
 
for metrics in runner.train(schedule=schedule):
    epoch = runner.current_epoch
 
    if epoch <= PHASE_0_EPOCHS:
        phase_label = "0"
    elif epoch <= PHASE_0_EPOCHS + PHASE_1_EPOCHS:
        phase_label = "0+1"
    else:
        phase_label = "0+1+2"
 
    logger.log_epoch(
        epoch         = epoch,
        loss_alpha    = metrics["loss_alpha"],
        loss_color    = metrics["loss_color"],
        loss_overflow = metrics["loss_overflow"],
        loss_total    = metrics["loss_total"],
        phase         = phase_label,
        model         = runner.model,
        is_final      = (epoch == TOTAL_EPOCHS),
    )
 
    if epoch % 500 == 0:
        W    = runner.W
        dist = dict(Counter(runner.pool_tasks))
        print(f"  [{epoch:>6}/{TOTAL_EPOCHS}]  "
              f"loss={metrics['loss_total']:.5f}  "
              f"color={metrics['loss_color']:.5f}  "
              f"overflow={metrics['loss_overflow']:.5f}  "
              f"pool={dist}  "
              f"W=(a={W['alpha']:.1f} c={W['color']:.1f} o={W['overflow']:.1f})")
 
print("\nTraining complete.")

In [ ]:
nca_cfg = NCAConfig(
    grid_size                 = tuple(config["grid"]["size"]),
    hidden_channels           = config["cell"]["hidden_channels"],
    visible_channels          = config["cell"]["visible_channels"],
    alive_threshold           = config["cell"]["alive_threshold"],
    task_channels             = config["cell"]["task_channels"],
    pos_channels              = config["cell"]["pos_channels"],
    perception_kernel_radius  = config["perception"]["kernel_radius"],
    perception_channel_groups = config["perception"]["channel_groups"],
    update_hidden_dim         = config["update"]["hidden_dim"],
    update_stochastic         = config["update"]["stochastic_update"],
    update_fire_rate          = config["update"]["fire_rate"],
)
nca_model = NCAModel(nca_cfg)
nca_model.grid.load_state_dict(runner.model.state_dict())
 
save_path = f"{OUTPUT_DIR}/model_final_run{logger.run_id}.pt"
nca_model.save(save_path)
print(f"NCAModel saved -> {save_path}")

In [ ]:
df = pd.read_csv(logger._loss_path)
 
fig, ax = plt.subplots(figsize=(12, 4.2))
ax.semilogy(df["epoch"], df["loss_alpha"],    lw=0.8, color="#1f77b4", label="alpha loss")
ax.semilogy(df["epoch"], df["loss_color"],    lw=0.8, color="#ff7f0e", label="color loss")
ax.semilogy(df["epoch"], df["loss_overflow"], lw=0.8, color="#2ca02c", label="overflow loss")
ax.semilogy(df["epoch"], df["loss_total"],    lw=1.4, color="#d62728", label="total loss")
 
for ep, label in [(BOUNDARY_1, f"+ {TASK_NAMES[1]}"), (BOUNDARY_2, f"+ {TASK_NAMES[2]}")]:
    ax.axvline(ep, color="grey", lw=1, ls="--", alpha=0.7)
    ax.text(ep + 60, 0.92, label,
            transform=ax.get_xaxis_transform(), color="grey", fontsize=8, va="top")
 
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss (log scale)")
ax.set_title(
    f"Curriculum joint training - 3D-NCA  "
    f"[run {logger.run_id}, {N_TASKS} tasks, {TOTAL_EPOCHS} epochs]  "
    f"{'->'.join(TASK_NAMES)}"
)
ax.legend(loc="upper right", fontsize=8, framealpha=0.7)
ax.grid(True, which="both", alpha=0.25)
plt.tight_layout()
plt.savefig(f"{logger.run_dir}/loss_curve.png", dpi=200)
plt.show()

In [ ]:
SNAP_AT = [0, 16, 48, 96, 160, 256]
VIS     = config["cell"]["visible_channels"]
THR     = config["cell"]["alive_threshold"]
 
 
def run_with_snaps(state, snap_at):
    snaps, done = [], 0
    with torch.no_grad():
        for target_step in snap_at:
            n = target_step - done
            if n > 0:
                state = runner.model(state, steps=n, use_checkpointing=False)
                state = torch.clamp(state, -1.0, 1.0)
                done  = target_step
            snaps.append(state.detach().cpu())
    return state.detach(), snaps
 
 
runner.model.eval()
 
state = runner._new_seed(0)
state, snaps_0 = run_with_snaps(state, SNAP_AT)
 
state = runner._relabel(state, 1)
state, snaps_1 = run_with_snaps(state, SNAP_AT)
 
state = runner._relabel(state, 2)
state, snaps_2 = run_with_snaps(state, SNAP_AT)
 
all_phase_snaps = [snaps_0, snaps_1, snaps_2]
 
print("Inference alive voxels at final step:")
for name, snaps in zip(TASK_NAMES, all_phase_snaps):
    alive = int((snaps[-1][0, -1] > THR).sum().item())
    print(f"  {name:12s}: {alive:>5}")

In [ ]:
N_SNAPS = len(SNAP_AT)
fig = plt.figure(figsize=(N_SNAPS * 2.6, N_TASKS * 3.0))
gs  = gridspec.GridSpec(N_TASKS, N_SNAPS, figure=fig, wspace=0.04, hspace=0.28)
 
for row, (phase_name, snaps) in enumerate(zip(PHASE_NAMES, all_phase_snaps)):
    for col, (step, snap) in enumerate(zip(SNAP_AT, snaps)):
        ax = fig.add_subplot(gs[row, col], projection="3d")
        show_state_rgba_mpl(snap, visible_channels=VIS,
                            threshold=0.20, surface_only=True,
                            point_size=12, view_angle=(28, 40),
                            ax=ax, show=False, title="")
        ax.set_title(f"+{step}", fontsize=7, pad=1)
        if col == 0:
            ax.text2D(-0.22, 0.5, phase_name,
                      transform=ax.transAxes, fontsize=8, fontweight="bold",
                      va="center", ha="right", rotation=90)
 
fig.suptitle(
    f"Metamorphogenesis: {'->'.join(TASK_NAMES)}  [run {logger.run_id}]",
    fontsize=11, y=1.02,
)
plt.tight_layout()
out = f"{logger.run_dir}/morph_progression.png"
fig.savefig(out, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved -> {out}")

In [ ]:
fig, axes = plt.subplots(2, N_TASKS, figsize=(N_TASKS * 4.5, 9.0),
                         subplot_kw={"projection": "3d"})
 
for col, (name, tgt_rgba, snaps) in enumerate(zip(TASK_NAMES, TARGETS, all_phase_snaps)):
    n_t = show_volume_rgba_mpl(tgt_rgba, ax=axes[0, col],
                               surface_only=True, threshold=0.15,
                               point_size=35, view_angle=(28, 40),
                               show=False, title="")
    axes[0, col].set_title(f"Target\n{name}  ({n_t} vx)", fontsize=9)
 
    n_s = show_state_rgba_mpl(snaps[-1], visible_channels=VIS, ax=axes[1, col],
                              surface_only=True, threshold=0.20,
                              point_size=35, view_angle=(28, 40),
                              show=False, title="")
    axes[1, col].set_title(f"Output\n{name}  ({n_s} vx)", fontsize=9)
 
for row, label in enumerate(["Target", "Model output"]):
    axes[row, 0].text2D(-0.18, 0.5, label,
                        transform=axes[row, 0].transAxes,
                        fontsize=10, fontweight="bold",
                        va="center", ha="right", rotation=90)
 
fig.suptitle(f"Target vs model output  [run {logger.run_id}]", fontsize=12, y=1.01)
plt.tight_layout()
out = f"{logger.run_dir}/comparison.png"
fig.savefig(out, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved -> {out}")
 
print(f"\n-- Done -------------------------------------------------------")
print(f"  Run dir : {logger.run_dir}")
print(f"  Model   : {save_path}")
print(f"  Targets : {'->'.join(TASK_NAMES)}")

In [ ]:
!zip -r /kaggle/working/morph_creature_final.zip /kaggle/working/nca_morph